## Implement RAG with the Claude Messages API

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2 openai==2.38.0 qdrant-client==1.14.3

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# load claude model configurations
claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

# load openai model configurations
openai_api_key = os.getenv("OPENAI_API_KEY")

# Declare QDrant DB Configurations
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "margies_travel_embeddings"

### Creating the Anthropic Client

In [ ]:
import anthropic

anthropic_client = anthropic.Anthropic(api_key = claude_api_key)

### Creating the OpenAI Client

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    api_key = openai_api_key
)

### Create the Embedding Generator Function

In [ ]:
def generate_vector_embeddings(user_query):

    response = openai_client.embeddings.create(
        model = "text-embedding-3-small",
        input = user_query
    )

    return response.data[0].embedding

### Create the Context Retrieval Function

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

def context_retrieval(user_query):

    # Generating Vector Embeddings for the User Query
    user_embeddings = generate_vector_embeddings(user_query)

    # Initialize Qdrant client
    qdrant_client = QdrantClient(url=QDRANT_URL)

    # Throwing a request to QDrantDB to retrieve the top-matched content along with the vector embeddings
    search_result = qdrant_client.query_points(
        collection_name = "margies_travel_embeddings",
        query = user_embeddings,
        with_vectors=True,
        with_payload=True,
        limit=2
    ).points

    # Retrieving the top text content that can be used to answer the user query
    supporting_text_content = search_result[0].payload.get("content")
    return supporting_text_content


### Summarize with the Claude Messages API

In [ ]:
user_query = """What hotels are offered by Margies Travels in San Francisco ?"""

context = context_retrieval(user_query)

with anthropic_client.messages.stream(
    model=claude_model_name,
    system = "You are a helpful AI Chatbot",
    messages=[
        {
            "role": "user",
            "content":  f"""answer the user query using the provided supporting knowledge \n 
                                            user query: {user_query} \n
                                            supporting_text: {context}""",
        }
    ],
    max_tokens=10000
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)